# Clinic activity D1 — functional trial tự chứa

Import notebook này vào **Kaggle CPU / Internet On**, rồi **Run All**. Không cần upload repo, commit/push hoặc chạy notebook Q6 cũ. Source D1 dirty overlay đã nhúng và hash; mỗi lượt tạo trial riêng.

Notebook kiểm/cài prerequisite thiếu, dùng Node24.16/PostgreSQL16/Nginx, tạo DB/roles/schema mới có guard, áp migration hiện có, seed10.000 lịch giả, kiểm18 PG parity/timezone cases và HTTP qua Nginx→Core→Clinic với DB thật. Có kiểm current-role/cache/TTL và snapshot read-only RepeatableRead.

**Không benchmark hoặc summary V2.** Core/Clinic composition chức năng dùng DB thật nhưng không chạy background broker/Redis/worker; không là full deployment hoặc D0 environment PASS. Chưa chứng minh quota riêng stack/generator. Secrets/DB/logs ở `/tmp`, ZIP kết quả chỉ chứa allowlist báo cáo. Run lỗi vẫn xuất báo cáo FAIL; không tự retry/reset hay dùng DB có sẵn.

In [ ]:
RUN_TRIAL = True
RUN_BENCHMARK = False
assert RUN_TRIAL is True and RUN_BENCHMARK is False, 'Notebook này chỉ hỗ trợ functional D1.'
import base64, hashlib, importlib.util, io, json, os, platform, stat, uuid
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
assert platform.system() == 'Linux' and Path('/kaggle/working').is_dir(), 'Chạy trong notebook Kaggle định dùng.'
assert os.geteuid() == 0, 'Setup native cần quyền root trong kernel trial.'

SOURCE_SHA256 = '770fd6b8a135b66084ffb91d215abc914e0c02c1af55e0e3e073aa324f28e5d4'
SOURCE_BASE64 = '...'
raw = base64.b64decode(SOURCE_BASE64, validate=True)
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256, 'Source archive hash mismatch'
# Bootstrap only the validator from the globally hash-verified archive; no cell
# or dependency from a previous notebook is imported.
with ZipFile(io.BytesIO(raw)) as z:
    validator = z.read('source/benchmarks/phase2/kaggle/make_activity_notebook.py')
namespace = {'__name__': 'activity_bundle_validator', '__file__': '/tmp/activity-validator/benchmarks/phase2/kaggle/make_activity_notebook.py'}
exec(compile(validator, '<verified-activity-validator>', 'exec'), namespace)
source_manifest, values = namespace['verified_members'](raw, SOURCE_SHA256)
RUN = str(uuid.uuid4())
PRIVATE = Path('/tmp') / ('clinic-activity-d1-' + RUN)
OUTPUT = Path('/kaggle/working') / ('clinic-activity-d1-' + RUN)
PRIVATE.mkdir(mode=0o700); OUTPUT.mkdir(mode=0o700)
with (PRIVATE/'source.zip').open('xb') as handle:
    handle.write(raw)
ROOT = PRIVATE / 'source'; ROOT.mkdir(mode=0o700)
for name, value in values.items():
    destination = ROOT.joinpath(*PurePosixPath(name).parts)
    assert destination.resolve().is_relative_to(ROOT.resolve())
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('xb') as handle:
        handle.write(value)
spec = importlib.util.spec_from_file_location('activity_native', ROOT/'benchmarks/phase2/kaggle/activity_native.py')
runner = importlib.util.module_from_spec(spec); spec.loader.exec_module(runner)
print({'run': RUN, 'source_verified': True, 'source_archive_sha256': SOURCE_SHA256, 'benchmark': False})

## Chạy functional trial
Mỗi stage báo tiến độ. Dừng khi lỗi; private logs giữ trong `/tmp`, không copy log/context/token vào chat. Cleanup chỉ dừng process của trial mới; không xóa dữ liệu hoặc stop dịch vụ cũ.

In [ ]:
summary = runner.run_trial(ROOT, PRIVATE, OUTPUT, RUN, SOURCE_SHA256)
print(json.dumps(summary, indent=2))
from IPython.display import display, FileLink
display(FileLink(summary['download']))

## Gửi ZIP kết quả
Tải file `.zip` ở link trên rồi gửi để kiểm evidence. PASS_IN_FUNCTIONAL_SCOPE chỉ cho các ca đã chạy; performance vẫn NOT_RUN và D0 vẫn PARTIAL. Không gửi file context/credentials/DB/log riêng. Nếu FAIL, giữ ZIP lỗi để chẩn đoán trước khi chạy trial mới.